In [29]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/MindCache")
sys.path.insert(0, str(PROJECT_ROOT))

import os
os.chdir(PROJECT_ROOT)

print("Files:", os.listdir(PROJECT_ROOT))

from Database.db_setup import engine, Topic
print("Import success")

Files: ['Memory_extract', 'Database', 'output.txt', 'embedder.ipynb', 'nodes_description_only.py', 'mindcache.db-wal', 'fix_uncategorized.py', 'project_description.md', 'embedder.py', 'mindcache.db-shm', 'reorganize_tree.py', 'Modelfile', 'reembed_roots.py', 'rebuild_tree_from_memories.py', 'mindcache.db']
Import success


In [31]:
import gc
import numpy as np
import torch
from sqlalchemy.orm import sessionmaker
from sentence_transformers import SentenceTransformer

from Database.db_setup import engine, Topic

MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
MAX_SEQ_LENGTH = 2048

if torch.cuda.is_available():
    DEVICE = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

    if total_vram_gb >= 20:
        BATCH_SIZE = 96
    elif total_vram_gb >= 14:
        BATCH_SIZE = 64
    elif total_vram_gb >= 10:
        BATCH_SIZE = 48
    else:
        BATCH_SIZE = 24

    torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    DEVICE = "cpu"
    gpu_name = "CPU"
    total_vram_gb = 0
    BATCH_SIZE = 8

DB_BATCH_SIZE = BATCH_SIZE

print(f"Embedding model: {MODEL_NAME}")
print(f"Device: {DEVICE} ({gpu_name})")
if DEVICE == "cuda":
    print(f"GPU memory: {total_vram_gb:.1f} GB")
else:
    print("CUDA not detected. This notebook will run on CPU and be slower.")
print(f"Embedding batch size: {BATCH_SIZE}")

class EmbeddingManager:
    def __init__(self):
        print(f"Loading embedding model on {DEVICE}...")
        self.model = SentenceTransformer(
            MODEL_NAME,
            trust_remote_code=True,
            device=DEVICE,
        )
        self.model.max_seq_length = MAX_SEQ_LENGTH
        self.batch_size = BATCH_SIZE
        self.model.eval()
        print("Model loaded.")

    def get_batch_embeddings(self, text_list):
        if not text_list:
            return []

        instruction = "Instruct: Represent this text for retrieval so it can be retrieved accurately"

        while True:
            try:
                with torch.inference_mode():
                    return self.model.encode(
                        text_list,
                        prompt=instruction,
                        batch_size=self.batch_size,
                        normalize_embeddings=True,
                        convert_to_numpy=True,
                        show_progress_bar=False,
                    )
            except RuntimeError as exc:
                is_oom = "out of memory" in str(exc).lower()
                if DEVICE == "cuda" and is_oom and self.batch_size > 4:
                    self.batch_size = max(4, self.batch_size // 2)
                    torch.cuda.empty_cache()
                    print(f"CUDA OOM. Retrying with batch_size={self.batch_size}...")
                    continue
                raise

    @staticmethod
    def to_blob(vector):
        return np.asarray(vector, dtype=np.float32).tobytes()

def normalize_text(text):
    return " ".join((text or "").split())


Embedding model: Qwen/Qwen3-Embedding-0.6B
Device: cuda (Tesla T4)
GPU memory: 14.6 GB
Embedding batch size: 64


In [32]:
def run_embedding_job():
    Session = sessionmaker(bind=engine)
    session = Session()
    embedder = EmbeddingManager()

    try:
        topic_ids = [
            topic_id
            for (topic_id,) in (
                session.query(Topic.id)
                .filter(
                    Topic.description != None,
                    Topic.embedding == None,
                )
                .order_by(Topic.id)
                .all()
            )
        ]

        total_topics = len(topic_ids)
        print(f"Found {total_topics} topics needing vectors.")

        for start in range(0, total_topics, DB_BATCH_SIZE):
            batch_ids = topic_ids[start : start + DB_BATCH_SIZE]
            batch = (
                session.query(Topic)
                .filter(Topic.id.in_(batch_ids))
                .order_by(Topic.id)
                .all()
            )

            texts = [normalize_text(topic.description) for topic in batch]
            vectors = embedder.get_batch_embeddings(texts)

            for topic, vector in zip(batch, vectors):
                topic.embedding = embedder.to_blob(vector)

            session.commit()
            session.expunge_all()

            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            gc.collect()

            print(f"Processed {start + len(batch)}/{total_topics} topics...")

        print("Embedding job complete.")
    except Exception:
        session.rollback()
        raise
    finally:
        session.close()


In [ ]:
run_embedding_job()

Loading embedding model on cuda...


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Model loaded.
Found 0 topics needing vectors.
Embedding job complete.


In [39]:
import sqlite3

conn = sqlite3.connect("/content/drive/MyDrive/MindCache/mindcache.db")
cursor = conn.cursor()

cursor.execute("SELECT COUNT(*) FROM topics WHERE embedding IS NULL")
print(cursor.fetchone())

(0,)


In [40]:
import sqlite3

db_path = "/content/drive/MyDrive/MindCache/mindcache.db"

conn = sqlite3.connect(db_path)
conn.execute("PRAGMA wal_checkpoint(FULL);")
conn.commit()
conn.close()

print("WAL merged into main DB")

WAL merged into main DB
